# F04.019 — Comparativo de extratores (cada um SOZINHO)

Compara **Docling**, **Marker 2**, **MinerU** e **olmOCR 2** no mesmo PDF, sem híbrido,
sem rede de segurança do PyMuPDF: **um modelo por página**.

**Como rodar:** `Runtime -> GPU`. Rode a célula de config, depois **uma seção por vez**.
Cada extrator salva o resultado em disco — se um falhar ou exigir restart do runtime,
os outros não se perdem. Ao final, a célula de comparação lê tudo que existir.

> As 4 bibliotecas disputam versões de `torch`/`transformers`. Se aparecer erro de
> import, use `Runtime -> Restart session` e rode **só** a seção do extrator em questão.


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
import os, json, time, gc, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE = Path('/content/drive/MyDrive/Pdfextractor')
PDF  = BASE/'data'/'PDFs concluídos'/'Bernardi (2022) - Particle films - Fernanda Bochi Dos Santos.pdf'
OUT  = BASE/'data'/'COMPARATIVO_EXTRATORES'
OUT.mkdir(parents=True, exist_ok=True)

assert PDF.exists(), f"PDF não encontrado: {PDF}"
print("PDF :", PDF.name)
print("OUT :", OUT)

def salvar(nome, ok, tempo_s, md_txt="", n_blocos=None, n_tabelas=None,
           tem_bbox=None, erro=None):
    """Guarda o resultado de um extrator (e o markdown completo ao lado)."""
    r = {'extrator': nome, 'ok': ok, 'tempo_s': round(tempo_s, 1),
         'chars': len(md_txt or ''), 'blocos': n_blocos, 'tabelas': n_tabelas,
         'tem_bbox': tem_bbox, 'erro': str(erro) if erro else None}
    (OUT/f'{nome}.json').write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding='utf-8')
    if md_txt:
        (OUT/f'{nome}.md').write_text(md_txt, encoding='utf-8')
    print(f"\n{'OK ' if ok else 'FALHOU'} {nome}: {r['tempo_s']}s | {r['chars']:,} chars"
          f" | blocos={n_blocos} | tabelas={n_tabelas}")
    if erro:
        print("   erro:", str(erro)[:300])
    return r

def limpar_memoria():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass


## 1. Docling (baseline atual)

In [ ]:
!pip install -q docling

In [ ]:
# ── DOCLING ─────────────────────────────────────────────────────────────────
try:
    from docling.document_converter import DocumentConverter

    t0 = time.time()
    doc = DocumentConverter().convert(str(PDF)).document
    md_txt = doc.export_to_markdown()
    dt = time.time() - t0

    itens = list(doc.iterate_items())
    tabelas = sum(1 for it, _ in itens if type(it).__name__ == 'TableItem')
    tem_bbox = any(getattr(it, 'prov', None) for it, _ in itens)
    salvar('docling', True, dt, md_txt, len(itens), tabelas, tem_bbox)
except Exception as e:
    salvar('docling', False, 0, erro=e)
limpar_memoria()


## 2. Marker 2 (datalab — mesma casa do Chandra)

In [ ]:
!pip install -q marker-pdf

In [ ]:
# ── MARKER 2 ────────────────────────────────────────────────────────────────
try:
    from marker.converters.pdf import PdfConverter
    from marker.models import create_model_dict
    from marker.output import text_from_rendered

    conv = PdfConverter(artifact_dict=create_model_dict())
    t0 = time.time()
    rendered = conv(str(PDF))
    dt = time.time() - t0

    md_txt, _, _ = text_from_rendered(rendered)
    meta = getattr(rendered, 'metadata', {}) or {}
    blocos = meta.get('block_counts') or meta.get('page_stats')
    n_blocos = len(blocos) if isinstance(blocos, list) else None
    tabelas = md_txt.count('<table') + md_txt.count('|---')
    salvar('marker2', True, dt, md_txt, n_blocos, tabelas, tem_bbox=True)
except Exception as e:
    salvar('marker2', False, 0, erro=e)
limpar_memoria()


## 3. MinerU

In [ ]:
!pip install -q "mineru[core]"

In [ ]:
# ── MINERU (via CLI, que é a interface estável) ─────────────────────────────
try:
    dest = OUT/'_mineru'
    dest.mkdir(exist_ok=True)
    t0 = time.time()
    p = subprocess.run(['mineru', '-p', str(PDF), '-o', str(dest)],
                       capture_output=True, text=True, timeout=3600)
    dt = time.time() - t0
    if p.returncode != 0:
        raise RuntimeError((p.stderr or p.stdout)[-500:])

    mds = sorted(dest.rglob('*.md'), key=lambda f: f.stat().st_size, reverse=True)
    md_txt = mds[0].read_text(encoding='utf-8') if mds else ''
    jsons = list(dest.rglob('*content_list*.json')) + list(dest.rglob('*middle*.json'))
    n_blocos = None
    if jsons:
        dados = json.loads(jsons[0].read_text(encoding='utf-8'))
        n_blocos = len(dados) if isinstance(dados, list) else None
    salvar('mineru', True, dt, md_txt, n_blocos,
           md_txt.count('<table') + md_txt.count('|---'), tem_bbox=bool(jsons))
except Exception as e:
    salvar('mineru', False, 0, erro=e)
limpar_memoria()


## 4. olmOCR 2 (AI2)

Baixa um modelo grande e usa vLLM — é a seção mais pesada. Se estourar a memória,
rode-a sozinha após `Restart session`.

In [ ]:
!pip install -q olmocr[gpu]

In [ ]:
# ── olmOCR 2 ────────────────────────────────────────────────────────────────
try:
    ws = OUT/'_olmocr'
    ws.mkdir(exist_ok=True)
    t0 = time.time()
    p = subprocess.run([sys.executable, '-m', 'olmocr.pipeline', str(ws),
                        '--markdown', '--pdfs', str(PDF)],
                       capture_output=True, text=True, timeout=7200)
    dt = time.time() - t0
    if p.returncode != 0:
        raise RuntimeError((p.stderr or p.stdout)[-500:])

    mds = sorted(ws.rglob('*.md'), key=lambda f: f.stat().st_size, reverse=True)
    md_txt = mds[0].read_text(encoding='utf-8') if mds else ''
    if not md_txt:                                  # fallback: saída em JSONL
        partes = []
        for jf in ws.rglob('*.jsonl'):
            for linha in jf.read_text(encoding='utf-8').splitlines():
                try:
                    partes.append(json.loads(linha).get('text', ''))
                except Exception:
                    pass
        md_txt = "\n\n".join(partes)
    salvar('olmocr2', True, dt, md_txt, None,
           md_txt.count('<table') + md_txt.count('|---'), tem_bbox=False)
except Exception as e:
    salvar('olmocr2', False, 0, erro=e)
limpar_memoria()


## 5. Comparação

Lê tudo que foi salvo e mede: tempo, volume de texto, tabelas e **qualidade** —
duplicação de parágrafo e presença de trechos-chave que sabemos existir neste PDF.

In [ ]:
# ── COMPARAÇÃO ──────────────────────────────────────────────────────────────
import re

def canon(t):
    return re.sub(r'[^a-z0-9]', '', (t or '').lower())

# trechos que EXISTEM no PDF (p. 4) — servem de teste de recall
TRECHOS = [
    ("CAT/Kraus",  "activity of CAT was determined according to Kraus"),
    ("SOD/Giann.", "SOD activity was determined according to Giannopolitis"),
    ("coef. 39.4", "extinction coefficient of 39.4"),
    ("ANOVA",      "Analysis of variance"),
]

def duplicacao(txt):
    """% de linhas longas repetidas — detecta o parágrafo saindo 2x."""
    linhas = [canon(l) for l in (txt or '').splitlines()]
    linhas = [l for l in linhas if len(l) > 60]
    if not linhas:
        return 0.0
    return round(100 * (1 - len(set(linhas)) / len(linhas)), 1)

linhas_tab = []
for jf in sorted(OUT.glob('*.json')):
    r = json.loads(jf.read_text(encoding='utf-8'))
    mdf = OUT/f"{r['extrator']}.md"
    txt = mdf.read_text(encoding='utf-8') if mdf.exists() else ''
    c = canon(txt)
    achou = sum(1 for _, frag in TRECHOS if canon(frag) in c)
    r['recall'] = f"{achou}/{len(TRECHOS)}"
    r['dup_%'] = duplicacao(txt)
    r['s_por_pag'] = round(r['tempo_s'] / 12, 1) if r['tempo_s'] else 0
    linhas_tab.append(r)

print(f"{'extrator':<10} {'ok':<4} {'tempo':>8} {'s/pág':>7} {'chars':>9} "
      f"{'tabelas':>8} {'recall':>7} {'dup%':>6}")
print("-" * 68)
for r in sorted(linhas_tab, key=lambda x: (not x['ok'], x['tempo_s'])):
    print(f"{r['extrator']:<10} {str(r['ok']):<4} {r['tempo_s']:>7.1f}s {r['s_por_pag']:>7} "
          f"{r['chars']:>9,} {str(r['tabelas']):>8} {r['recall']:>7} {r['dup_%']:>6}")

print("\nBaseline do pipeline híbrido atual: 296.7s | 87,300 chars | 363 blocos")
print("\nrecall = trechos conhecidos da p.4 encontrados | dup% = linhas longas repetidas")
for r in linhas_tab:
    if not r['ok']:
        print(f"\n{r['extrator']} falhou: {(r['erro'] or '')[:400]}")


In [ ]:
# ── amostra lado a lado (mesmo trecho em cada extrator) ─────────────────────
ALVO = "activity of CAT was determined"
for jf in sorted(OUT.glob('*.json')):
    nome = json.loads(jf.read_text(encoding='utf-8'))['extrator']
    f = OUT/f'{nome}.md'
    if not f.exists():
        continue
    txt = f.read_text(encoding='utf-8')
    i = txt.find(ALVO)
    print("=" * 72)
    print(nome.upper())
    print(txt[i-200:i+420].strip() if i >= 0 else "(trecho não encontrado)")
    print()
